In [ ]:
from google.colab import files
uploaded=files.upload()

Saving scan.xml to scan (1).xml


In [ ]:
import xml.etree.ElementTree as ET
tree=ET.parse('scan.xml')
root=tree.getroot()
print(root.tag)

nmaprun


In [ ]:
for host in root.findall("host"):
    print("Host Found")

Host Found


In [ ]:
for host in root.findall("host"):
    for port in host.findall(".//port"):
        port_id=port.get("portid")
        print("port:",port_id)

port: 22
port: 25
port: 80
port: 135
port: 139
port: 445
port: 9929
port: 31337


In [ ]:
for host in root.findall("host"):
    for port in host.findall(".//port"):
        port_id=port.get("portid")
        service=port.find("service")
        if service is not None:
            service_name=service.get("name")
            print(port_id, ":",service_name)

22 : ssh
25 : smtp
80 : http
135 : msrpc
139 : netbios-ssn
445 : microsoft-ds
9929 : nping-echo
31337 : tcpwrapped


In [ ]:
for host in root.findall("host"):
    for port in host.findall(".//port"):
        port_id=port.get("portid")
        service=port.find("service")
        service_name=service.get("name")
        product=service.get("product")
        if product is not None:
            print(port_id, ":",product,":",service_name)

22 : OpenSSH : ssh
80 : Apache httpd : http
9929 : Nping echo : nping-echo


In [ ]:
import requests
import base64
from google.colab import userdata
API_KEY=userdata.get('API_KEY')
BASE_URL=f"https://www.virustotal.com/api/v3/ip_addresses/45.33.32.156"
headers={
    "x-apikey":API_KEY
}
response=requests.get(BASE_URL,headers=headers)
data=response.json()
print(data.keys())


dict_keys(['data'])


In [ ]:
print(data['data'].keys())

dict_keys(['id', 'type', 'links', 'attributes'])


In [ ]:
print(type(data['data']['id']))

<class 'str'>


In [ ]:
print(type(data['data']['attributes']))

<class 'dict'>


In [ ]:
print(len(data['data']['id']))

12


In [ ]:
for i in data['data']['attributes']:
  print(i)

rdap
last_analysis_date
continent
country
as_owner
last_modification_date
reputation
whois_date
network
asn
whois
total_votes
tags
last_analysis_stats
regional_internet_registry
last_analysis_results


In [ ]:
attributes=data['data']['attributes']
stats=attributes["last_analysis_stats"]

print("malicious:",stats["malicious"])
print("suspicious:",stats["suspicious"])
print("harmless:",stats["harmless"])

malicious: 4
suspicious: 1
harmless: 55


In [ ]:
if stats['malicious']>0:
  print("malicious")
else:
  print("not malicious")

malicious


In [ ]:
malicious_count=stats["malicious"]
if malicious_count>10:
  print("high risk file")
elif malicious_count>0:
  print("medium risk")
else:
  print("low risk")

medium risk


In [ ]:
!apt-get install nmap -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
nmap is already the newest version (7.91+dfsg1+really7.80+dfsg1-2ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [ ]:
!pip install python-nmap

In [ ]:
import nmap

In [ ]:
nm=nmap.PortScanner()
nm.scan('localhost','1-10000',arguments="-sV")

for host in nm.all_hosts():
  print(f"host:{host}")
  for protocol in nm[host].all_protocols():
    print(f"protocol:{protocol}")
    print(f"ports:{nm[host][protocol].keys()}")

host:127.0.0.1
protocol:tcp
ports:dict_keys([3453, 8080])


In [ ]:
for host in nm.all_hosts():
  print(f"host:{host}")
  for protocol in nm[host].all_protocols():
    print(f"protocol:{protocol}")
    ports=nm[host][protocol].keys()
    for port in ports:
      state=nm[host][protocol][port]['state']
      service=nm[host][protocol][port]['name']
      print(f"port:{port} -> state:{state} -> service:{service}")

host:127.0.0.1
protocol:tcp
port:3453 -> state:open -> service:http
port:8080 -> state:open -> service:http-proxy


In [ ]:
for port in ports:
      product=nm[host][protocol][port].get("product")
      version=nm[host][protocol][port].get("version")
      print(f"port:{port} -> state:{state} -> service:{service} -> product:{product} -> version:{version}")

port:3453 -> state:open -> service:http-proxy -> product:Tornado httpd -> version:6.5.1
port:8080 -> state:open -> service:http-proxy -> product: -> version:
